In [1]:
# Import Packages
import numpy as np
import matplotlib.pyplot as plt

import qutip as q
from tqdm.notebook import tqdm

####################################################################
# Import utilities
import sys
from pathlib import Path
import qutip as q
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

from utilities.functions import cat, sqv, pssqv, sq_cat
from utilities.artificial_samples import sample_homodyne
from utilities.MLE_class import MLE
from utilities.MLE_class_faster import MLE as MLE_f
from utilities.plotting import plot_dm, plot_Wigner, plot_fidelities
from utilities.benchmarking import diff_samples

In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


# GLOBAL SETTINGS

X_RANGE = (-5, 5)

# Grid used for the displayed Wigner function
X_POINTS = 200

# Grid used internally by MLE
MLE_X_POINTS = 400

# Maximum number of MLE iterations
MAX_ITER = 100

# Number of homodyne samples for each angle
N_SAMPLES_THETA = 1000

# Available widget choices
ANGLE_OPTIONS = [2, 3, 4, 6, 8, 10, 12]
BIN_OPTIONS = [10, 20, 30, 40, 50]

CUTOFF_OPTIONS = [
    15, 20, 25, 30, 35, 40,
    45, 50, 55, 60, 65, 70,
    75, 80
]

# Wigner-function coordinate grid
xvec = np.linspace(
    X_RANGE[0],
    X_RANGE[1],
    X_POINTS
)



# STATE GENERATOR


def make_superposition(n, parity, cutoff):
    """
    Generate

        (|0> + (-1)^parity |n>) / sqrt(2)
    """

    if n >= cutoff:
        raise ValueError(
            f"n={n} must be smaller than cutoff={cutoff}"
        )

    ket = np.zeros(cutoff, dtype=complex)

    ket[0] = 1 / np.sqrt(2)
    ket[n] = (-1) ** parity / np.sqrt(2)

    return qt.Qobj(ket)



# TARGET STATES


state_options = {

    "|0> + |1>": (1, 0),
    "|0> - |1>": (1, 1),

    "|0> + |2>": (2, 0),
    "|0> - |2>": (2, 1),

    "|0> + |3>": (3, 0),
    "|0> - |3>": (3, 1),

    "|0> + |4>": (4, 0),
    "|0> - |4>": (4, 1),

    "|0> + |5>": (5, 0),
    "|0> - |5>": (5, 1),
}



# WIDGETS


state_dropdown = widgets.Dropdown(
    options=list(state_options.keys()),
    value="|0> - |3>",
    description="State:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="220px"
    )
)


angle_dropdown = widgets.Dropdown(
    options=ANGLE_OPTIONS,
    value=8,
    description="Angles:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="180px"
    )
)


bin_dropdown = widgets.Dropdown(
    options=BIN_OPTIONS,
    value=30,
    description="Bins:",
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="180px"
    )
)


cutoff_slider = widgets.IntSlider(
    value=15,
    min=min(CUTOFF_OPTIONS),
    max=max(CUTOFF_OPTIONS),
    step=5,
    description="Fock cutoff:",
    continuous_update=False,
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="350px"
    )
)


iteration_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=MAX_ITER,
    step=1,
    description="Iteration:",
    continuous_update=False,
    style={
        "description_width": "initial"
    },
    layout=widgets.Layout(
        width="350px"
    )
)



# ONE AND ONLY ONE OUTPUT AREA


output = widgets.Output()



# CACHE


results_cache = {}



# GET MLE RESULT


def get_mle_result():

    state_name = state_dropdown.value
    n_angles = angle_dropdown.value
    n_bins = bin_dropdown.value
    cutoff = cutoff_slider.value

    cache_key = (
        state_name,
        n_angles,
        n_bins,
        cutoff
    )

  ----
    # Use cached result if it exists
  ----

    if cache_key in results_cache:
        return results_cache[cache_key]

  ----
    # Generate target state
  ----

    n, parity = state_options[state_name]

    target_state = make_superposition(
        n=n,
        parity=parity,
        cutoff=cutoff
    )

  ----
    # Generate homodyne measurements
  ----

    thetas, samples = sample_homodyne(
        state=target_state,
        n_angles=n_angles,
        n_samples_theta=N_SAMPLES_THETA,
        bin_data=False,
        x_vec=np.linspace(
            X_RANGE[0],
            X_RANGE[1],
            MLE_X_POINTS
        )
    )

  ----
    # Create MLE object
  ----

    print(
        f"Running MLE for {state_name} | "
        f"angles={n_angles} | "
        f"bins={n_bins} | "
        f"cutoff={cutoff}"
    )

    mle = MLE(
        data=samples,
        N_bins=n_bins,
        N_cutoff=cutoff,
        x_lims=X_RANGE,
        x_points=MLE_X_POINTS
    )

  ----
    # Run MLE
  ----

    result = mle.run(
        max_iter=MAX_ITER,
        store_states=True
    )

  ----
    # Cache
  ----

    results_cache[cache_key] = (
        result,
        target_state
    )

    return result, target_state



# MAIN UPDATE FUNCTION


def update_display(change=None):

  ----
    # Completely clear previous output
  ----

    with output:

        clear_output(wait=True)

        try:

            # =================================================
            # GET CURRENT PARAMETERS
            # =================================================

            state_name = state_dropdown.value
            n_angles = angle_dropdown.value
            n_bins = bin_dropdown.value
            cutoff = cutoff_slider.value

            # =================================================
            # GET MLE RESULT
            # =================================================

            result, target_state = get_mle_result()

            # =================================================
            # DETERMINE NUMBER OF ITERATIONS
            # =================================================

            n_iterations = len(result.states) - 1

            # Make sure iteration slider is valid
            if iteration_slider.max != n_iterations:

                iteration_slider.unobserve(
                    update_display,
                    names="value"
                )

                iteration_slider.max = n_iterations

                if iteration_slider.value > n_iterations:
                    iteration_slider.value = n_iterations

                iteration_slider.observe(
                    update_display,
                    names="value"
                )

            # =================================================
            # CURRENT ITERATION
            # =================================================

            iteration = iteration_slider.value

            # =================================================
            # RECONSTRUCTED STATE
            # =================================================

            rho_mle = qt.Qobj(
                result.states[iteration]
            )

            # =================================================
            # FIDELITY
            # =================================================

            fidelity = qt.fidelity(
                target_state,
                rho_mle
            ) ** 2

            # =================================================
            # PREVIOUS ITERATION FIDELITY
            # =================================================

            if (
                iteration > 0
                and hasattr(result, "fidelities")
                and iteration - 1 < len(result.fidelities)
            ):

                convergence_fidelity = (
                    result.fidelities[iteration - 1]
                )

            else:

                convergence_fidelity = np.nan

            # =================================================
            # INFORMATION BOX
            # =================================================

            info_html = f"""

            <div style="
                border: 1px solid #cccccc;
                border-radius: 8px;
                padding: 12px 16px;
                margin-bottom: 15px;
                width: 420px;
                font-family: Arial, sans-serif;
                background-color: #fafafa;
            ">

                <h4 style="
                    margin-top: 0;
                    margin-bottom: 10px;
                ">
                    Reconstruction parameters
                </h4>

                <b>Target state:</b>
                {state_name}

                <br>

                <b>Homodyne angles:</b>
                {n_angles}

                <br>

                <b>Number of bins:</b>
                {n_bins}

                <br>

                <b>Fock cutoff:</b>
                {cutoff}

                <br>

                <b>MLE iteration:</b>
                {iteration}

                <hr>

                <b>Fidelity to original state:</b>

                <span style="
                    font-size: 18px;
                ">
                    {fidelity:.8f}
                </span>

                <br>

                <b>Fidelity to previous iteration:</b>
                {convergence_fidelity:.8f}

            </div>
            """

            display(
                HTML(info_html)
            )

            # =================================================
            # WIGNER FUNCTION
            # =================================================

            W = qt.wigner(
                rho_mle,
                xvec,
                xvec
            )

            # =================================================
            # SYMMETRIC COLOR SCALE
            # =================================================

            vmax = np.max(
                np.abs(W)
            )

            if vmax == 0:
                vmax = 1.0

            # =================================================
            # CREATE EXACTLY ONE FIGURE
            # =================================================

            fig, ax = plt.subplots(
                figsize=(7, 6)
            )

            # =================================================
            # WIGNER PLOT
            # =================================================

            contour = ax.contourf(
                xvec,
                xvec,
                W,
                levels=100,
                cmap="RdBu_r",
                vmin=-vmax,
                vmax=vmax
            )

            ax.set_xlabel(
                r"$x$"
            )

            ax.set_ylabel(
                r"$p$"
            )

            ax.set_title(
                "MLE Reconstructed Wigner Function",
                fontsize=14
            )

            # =================================================
            # COLORBAR
            # =================================================

            colorbar = fig.colorbar(
                contour,
                ax=ax
            )

            colorbar.set_label(
                r"$W(x,p)$"
            )

            # =================================================
            # LAYOUT
            # =================================================

            fig.tight_layout()

            # =================================================
            # DISPLAY FIGURE EXACTLY ONCE
            # =================================================

            display(fig)

            # Prevent Jupyter from displaying it again
            plt.close(fig)

        except Exception as error:

            # =================================================
            # ERROR DISPLAY
            # =================================================

            error_html = f"""

            <div style="
                border: 1px solid #cc0000;
                border-radius: 8px;
                padding: 12px;
                color: #990000;
                font-family: Arial, sans-serif;
            ">

                <b>Error:</b>

                <br><br>

                {type(error).__name__}: {error}

            </div>

            """

            display(
                HTML(error_html)
            )



# CALLBACKS


def parameter_changed(change):

    update_display()



# REGISTER CALLBACKS


state_dropdown.observe(
    parameter_changed,
    names="value"
)

angle_dropdown.observe(
    parameter_changed,
    names="value"
)

bin_dropdown.observe(
    parameter_changed,
    names="value"
)

cutoff_slider.observe(
    parameter_changed,
    names="value"
)

iteration_slider.observe(
    parameter_changed,
    names="value"
)



# USER INTERFACE


controls = widgets.VBox([

    widgets.HBox([
        state_dropdown,
        angle_dropdown,
        bin_dropdown
    ]),

    widgets.HBox([
        cutoff_slider,
        iteration_slider
    ])

])



# DISPLAY EVERYTHING ONCE


display(
    controls
)

display(
    output
)



# INITIAL UPDATE


update_display()

Output()

# Cat state

In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML



# GLOBAL SETTINGS


X_RANGE = (-6, 6)
X_POINTS = 200

N_SAMPLES_THETA = 1000
MAX_ITER = 100

ANGLE_OPTIONS = [4, 8, 12, 16, 20]
BIN_OPTIONS = [10, 20, 30, 40, 50]


# Phase-space grid
xvec = np.linspace(
    X_RANGE[0],
    X_RANGE[1],
    X_POINTS
)



# CAT STATE GENERATOR


def make_cat_state(cutoff, parity, alpha_abs, theta):
    """
    Generate a cat state.

    alpha = |alpha| exp(i theta)

    parity = 0 -> even cat
    parity = 1 -> odd cat
    """

    alpha = alpha_abs * np.exp(1j * theta)

    return cat(
        cutoff,
        parity,
        alpha
    )



# WIDGETS


# ------------------------------------------------------------
# Cat parity
# ------------------------------------------------------------

parity_dropdown = widgets.Dropdown(
    options=[
        ('Even cat (+)', 0),
        ('Odd cat (-)', 1)
    ],
    value=0,
    description='Parity:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)


# ------------------------------------------------------------
# |alpha|
# ------------------------------------------------------------

alpha_slider = widgets.FloatSlider(
    value=np.sqrt(5),
    min=0.5,
    max=3.0,
    step=0.1,
    description=r'|α|:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)


# ------------------------------------------------------------
# Phase of alpha
# ------------------------------------------------------------

theta_slider = widgets.FloatSlider(
    value=np.pi / 4,
    min=0,
    max=2 * np.pi,
    step=np.pi / 16,
    description=r'θ:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)


# ------------------------------------------------------------
# Number of homodyne angles
# ------------------------------------------------------------

angle_dropdown = widgets.Dropdown(
    options=ANGLE_OPTIONS,
    value=8,
    description='Angles:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)


# ------------------------------------------------------------
# Number of bins
# ------------------------------------------------------------

bin_dropdown = widgets.Dropdown(
    options=BIN_OPTIONS,
    value=30,
    description='Bins:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)


# ------------------------------------------------------------
# Fock cutoff
# ------------------------------------------------------------

cutoff_slider = widgets.IntSlider(
    value=15,
    min=15,
    max=80,
    step=5,
    description='Fock cutoff:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)


# ------------------------------------------------------------
# Iteration
# ------------------------------------------------------------

iteration_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=MAX_ITER,
    step=1,
    description='Iteration:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)



# OUTPUT


plot_output = widgets.Output()
info_output = widgets.Output()



# CACHE


results_cache = {}



# RUN MLE


def get_mle_result():

    parity = parity_dropdown.value
    alpha_abs = alpha_slider.value
    theta = theta_slider.value

    n_angles = angle_dropdown.value
    n_bins = bin_dropdown.value
    cutoff = cutoff_slider.value


  ----
    # Cache key
  ----

    cache_key = (
        parity,
        round(alpha_abs, 4),
        round(theta, 6),
        n_angles,
        n_bins,
        cutoff
    )


  ----
    # Return cached result
  ----

    if cache_key in results_cache:
        return results_cache[cache_key]


  ----
    # Generate target cat state
  ----

    target_state = make_cat_state(
        cutoff=cutoff,
        parity=parity,
        alpha_abs=alpha_abs,
        theta=theta
    )


  ----
    # Generate homodyne data
  ----

    print(
        f"Generating homodyne data:\n"
        f"Parity = {parity}\n"
        f"|alpha| = {alpha_abs:.2f}\n"
        f"theta = {theta:.3f}\n"
        f"Angles = {n_angles}"
    )


    thetas, samples = sample_homodyne(
        state=target_state,
        n_angles=n_angles,
        n_samples_theta=N_SAMPLES_THETA,
        bin_data=False,
        x_vec=np.linspace(
            X_RANGE[0],
            X_RANGE[1],
            400
        )
    )


  ----
    # Run MLE
  ----

    print(
        f"Running MLE:\n"
        f"Bins = {n_bins}\n"
        f"Cutoff = {cutoff}"
    )


    mle = MLE(
        data=samples,
        N_bins=n_bins,
        N_cutoff=cutoff,
        x_lims=X_RANGE,
        x_points=400
    )


    result = mle.run(
        max_iter=MAX_ITER,
        store_states=True
    )


  ----
    # Cache result
  ----

    results_cache[cache_key] = (
        result,
        target_state,
        samples,
        thetas
    )


    return results_cache[cache_key]



# UPDATE ITERATION RANGE


def update_iteration_range():

    try:

        result, target_state, samples, thetas = get_mle_result()

        n_iterations = len(result.states) - 1

        iteration_slider.max = n_iterations

        if iteration_slider.value > n_iterations:
            iteration_slider.value = n_iterations

    except Exception as e:

        print("Error:", e)



# PLOT WIGNER


def update_plot(*args):

    with plot_output:

        clear_output(wait=True)

        try:

            result, target_state, samples, thetas = get_mle_result()


            # Current parameters
            iteration = iteration_slider.value
            n_bins = bin_dropdown.value
            n_angles = angle_dropdown.value
            cutoff = cutoff_slider.value


            # ------------------------------------------------
            # Get MLE state
            # ------------------------------------------------

            rho_mle = qt.Qobj(
                result.states[iteration]
            )


            # ------------------------------------------------
            # Fidelity
            # ------------------------------------------------

            fidelity = qt.fidelity(
                target_state,
                rho_mle
            ) ** 2


            # ------------------------------------------------
            # Wigner
            # ------------------------------------------------

            W = qt.wigner(
                rho_mle,
                xvec,
                xvec
            )


            # ------------------------------------------------
            # Plot
            # ------------------------------------------------

            fig, ax = plt.subplots(
                figsize=(7, 6)
            )


            # Symmetric limits
            vmax = np.max(np.abs(W))


            im = ax.contourf(
                xvec,
                xvec,
                W,
                levels=100,
                cmap='RdBu_r',
                vmin=-vmax,
                vmax=vmax
            )


            ax.set_xlabel(r'$x$')
            ax.set_ylabel(r'$p$')


            ax.set_title(
                'MLE Reconstructed Cat-State Wigner Function',
                fontsize=14
            )


            cbar = plt.colorbar(
                im,
                ax=ax
            )

            cbar.set_label(
                r'$W(x,p)$'
            )


            plt.tight_layout()
            plt.show()


        except Exception as e:

            print("Error:", e)



# INFORMATION BOX


def update_info(*args):

    with info_output:

        clear_output(wait=True)

        try:

            result, target_state, samples, thetas = get_mle_result()


            parity = parity_dropdown.value
            alpha_abs = alpha_slider.value
            theta = theta_slider.value

            n_angles = angle_dropdown.value
            n_bins = bin_dropdown.value
            cutoff = cutoff_slider.value
            iteration = iteration_slider.value


            # ------------------------------------------------
            # Current MLE state
            # ------------------------------------------------

            rho_mle = qt.Qobj(
                result.states[iteration]
            )


            # ------------------------------------------------
            # Fidelity
            # ------------------------------------------------

            fidelity = qt.fidelity(
                target_state,
                rho_mle
            ) ** 2


            # ------------------------------------------------
            # Convergence fidelity
            # ------------------------------------------------

            if iteration > 0:

                convergence_fidelity = (
                    result.fidelities[iteration - 1]
                )

            else:

                convergence_fidelity = np.nan


            # ------------------------------------------------
            # Cat parameters
            # ------------------------------------------------

            alpha = (
                alpha_abs *
                np.exp(1j * theta)
            )


            # ------------------------------------------------
            # Information box
            # ------------------------------------------------

            parity_name = (
                "Even (+)"
                if parity == 0
                else
                "Odd (-)"
            )


            html = f"""
            <div style="
                border: 1px solid #cccccc;
                border-radius: 8px;
                padding: 12px;
                margin-top: 10px;
                width: 450px;
                font-family: Arial;
            ">

                <h4 style="margin-top:0;">
                    Cat-State Reconstruction
                </h4>

                <b>Parity:</b>
                {parity_name}<br>

                <b>|α|:</b>
                {alpha_abs:.3f}<br>

                <b>arg(α):</b>
                {theta:.3f} rad<br>

                <b>α:</b>
                {alpha.real:.3f}
                {alpha.imag:+.3f}i<br>

                <b>Homodyne angles:</b>
                {n_angles}<br>

                <b>Number of bins:</b>
                {n_bins}<br>

                <b>Fock cutoff:</b>
                {cutoff}<br>

                <b>MLE iteration:</b>
                {iteration}<br>

                <hr>

                <b>Fidelity to target:</b>

                <span style="
                    font-size:20px;
                    font-weight:bold;
                ">
                    {fidelity:.8f}
                </span>

                <br>

                <b>Fidelity to previous iteration:</b>
                {convergence_fidelity:.8f}

            </div>
            """


            display(
                HTML(html)
            )


        except Exception as e:

            print("Error:", e)


# PARAMETER UPDATE

def parameter_changed(*args):

    update_iteration_range()
    update_info()
    update_plot()


# CONNECT WIDGETS
parity_dropdown.observe(
    parameter_changed,
    names='value'
)

alpha_slider.observe(
    parameter_changed,
    names='value'
)

theta_slider.observe(
    parameter_changed,
    names='value'
)

angle_dropdown.observe(
    parameter_changed,
    names='value'
)

bin_dropdown.observe(
    parameter_changed,
    names='value'
)

cutoff_slider.observe(
    parameter_changed,
    names='value'
)


# Iteration only changes which reconstructed state
# is displayed.

iteration_slider.observe(
    lambda change: (
        update_info(),
        update_plot()
    ),
    names='value'
)


# DISPLAY


controls = widgets.VBox([

    widgets.HBox([
        parity_dropdown,
        angle_dropdown,
        bin_dropdown
    ]),

    widgets.HBox([
        alpha_slider,
        theta_slider
    ]),

    widgets.HBox([
        cutoff_slider,
        iteration_slider
    ])
])


display(
    controls,
    info_output,
    plot_output
)


# Initial plot
update_iteration_range()
update_info()
update_plot()

Output()

Output()

Generating homodyne data:
Parity = 0
|alpha| = 2.24
theta = 0.785
Angles = 8
Running MLE:
Bins = 30
Cutoff = 15


## cat state homodyne plots

In [10]:
# Example:
N_cutoff = 50
parity = 0
theta = np.pi/4
alpha = np.sqrt(5) * np.exp(1j*theta)

state = cat(N_cutoff, parity, alpha)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output



# HOMODYNE SAMPLING PARAMETERS


N_angles = 24
N_samples_theta = 1000

x_range = (-5, 5)
x_points = 400

x_array = np.linspace(
    x_range[0],
    x_range[1],
    x_points
)



# GENERATE HOMODYNE DATA


thetas, samples = sample_homodyne(
    state=state,
    n_angles=N_angles,
    n_samples_theta=N_samples_theta,
    bin_data=False,
    x_vec=x_array
)

samples = np.asarray(samples)
thetas = np.asarray(thetas)


print("Data shape:", samples.shape)
print("Theta shape:", thetas.shape)

Data shape: (24, 1000)
Theta shape: (24,)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output



# PARAMETERS


N_angles = 40
N_samples_theta = 2000

x_range = (-5.5, 5.5)
x_points = 500

x_array = np.linspace(
    x_range[0],
    x_range[1],
    x_points
)



# GENERATE HOMODYNE DATA


thetas, samples = sample_homodyne(
    state=state,
    n_angles=N_angles,
    n_samples_theta=N_samples_theta,
    bin_data=False,
    x_vec=x_array
)

thetas = np.asarray(thetas)
samples = np.asarray(samples)


print("samples shape:", samples.shape)
print("thetas shape:", thetas.shape)



# WIDGET


theta_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(thetas) - 1,
    step=1,
    description='LO phase ',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)


output = widgets.Output()

# ANGULAR WIDTH / JITTER


theta_width = 0.025   

# PLOT


def plot_homodyne(theta_index=0):

    theta_selected = thetas[theta_index]

    x_selected = samples[theta_index]

    with output:

        clear_output(wait=True)

        
        # FIGURE
        

        fig = plt.figure(
            figsize=(12, 9),
            constrained_layout=True
        )

        gs = fig.add_gridspec(
            2,
            1,
            height_ratios=[3, 2]
        )

    

        ax1 = fig.add_subplot(gs[0])

      
        # Spread each theta distribution over a small width
      

        rng = np.random.default_rng(42)

        theta_scatter = np.repeat(
            thetas,
            samples.shape[1]
        )

        x_scatter = samples.flatten()

        # Random horizontal displacement
        theta_jitter = rng.uniform(
            -theta_width,
            theta_width,
            size=theta_scatter.shape
        )

        theta_scatter_spread = (
            theta_scatter + theta_jitter
        )

      
        # All homodyne measurements
      

        ax1.scatter(
            theta_scatter_spread,
            x_scatter,
            s=4,
            alpha=0.12,
            linewidths=0
        )

      
        # Selected theta measurements
      

        selected_theta_jitter = rng.uniform(
            -theta_width,
            theta_width,
            size=len(x_selected)
        )

        ax1.scatter(
            theta_selected + selected_theta_jitter,
            x_selected,
            s=10,
            alpha=0.55,
            linewidths=0
        )

      
        # Actual selected theta
      

        ax1.axvline(
            theta_selected,
            linestyle='--',
            linewidth=1.5,
            alpha=0.8
        )

        
        # BOTTOM: MARGINAL DISTRIBUTION
        

        ax2 = fig.add_subplot(gs[1])

      
        # Histogram
    
        ax2.hist(
            x_selected,
            bins=200,
            density=True,
            alpha=0.75
        )



        x_kde = np.linspace(
            x_range[0],
            x_range[1],
            500
        )

        std = np.std(x_selected)
        n = len(x_selected)

        bandwidth = (
            1.06 *
            std *
            n**(-1/5)
        )

        bandwidth = max(
            bandwidth,
            1e-3
        )

        kde = np.zeros_like(x_kde)

        # Chunked calculation
        chunk_size = 500

        for start in range(
            0,
            len(x_selected),
            chunk_size
        ):

            x_chunk = x_selected[
                start:start + chunk_size
            ]

            kde += np.sum(
                np.exp(
                    -0.5 *
                    (
                        (x_kde[:, None] - x_chunk[None, :])
                        / bandwidth
                    )**2
                ),
                axis=1
            )

        kde /= (
            len(x_selected)
            * bandwidth
            * np.sqrt(2*np.pi)
        )

        ax2.plot(
            x_kde,
            kde,
            linewidth=2.5
        )

      
        # Labels
      

        ax2.set_xlabel(
            r'Quadrature $x_\theta$',
            fontsize=13
        )

        ax2.set_ylabel(
            r'$P(x_\theta)$',
            fontsize=13
        )

        ax2.set_title(
            rf'Marginal Distribution at '
            rf'$\theta={theta_selected:.3f}$ rad',
            fontsize=14
        )

        ax2.set_xlim(
            x_range
        )

        ax2.grid(
            alpha=0.2
        )

        plt.show()



# UPDATE


def update_plot(change):

    plot_homodyne(
        change['new']
    )


theta_slider.observe(
    update_plot,
    names='value'
)


# DISPLAY

display(
    theta_slider,
    output
)


# Initial plot
plot_homodyne(0)

samples shape: (40, 2000)
thetas shape: (40,)


IntSlider(value=0, continuous_update=False, description='LO phase ', layout=Layout(width='500px'), max=39, sty…

Output()